# RLO Experiment Analysis

This notebook analyzes and visualizes results from all experiments:
- Section 4.1: Image Classification
- Section 4.2: Vision-Language (LiT)
- Section 4.3: Diffusion Models
- Section 4.4: Language Modeling
- Ablation Studies

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 11

# Color scheme
COLORS = {
    'adamw': '#1f77b4',
    'lion': '#ff7f0e',
    'rlo': '#2ca02c',
    'rlo_lambda_a': '#d62728',
    'smooth_lifted_rlo': '#9467bd',
}

LABELS = {
    'adamw': 'AdamW',
    'lion': 'LION',
    'rlo': 'RLO',
    'rlo_lambda_a': 'RLO-λA',
    'smooth_lifted_rlo': 'SmoothLifted-RLO',
}

In [ ]:
def load_results(result_dir, pattern):
    """Load all JSON result files matching pattern"""
    results = {}
    result_path = Path(result_dir)
    for f in result_path.glob(pattern):
        with open(f) as fp:
            data = json.load(fp)
            opt = data.get('optimizer', f.stem.split('_')[-2])
            results[opt] = data
    return results

## 1. Section 4.1: Image Classification Results

In [ ]:
# Load classification results
RESULT_DIR = './results_classification'

models = ['resnet50', 'vit_s16', 'vit_b16']
classification_results = {}

for model in models:
    classification_results[model] = load_results(RESULT_DIR, f"{model}_*_results.json")
    print(f"{model}: {list(classification_results[model].keys())}")

In [ ]:
# Plot training curves for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model in zip(axes, models):
    results = classification_results.get(model, {})
    for opt, data in results.items():
        if 'history' in data and 'val_acc' in data['history']:
            val_acc = data['history']['val_acc']
            ax.plot(val_acc, label=f"{LABELS.get(opt, opt)} ({data['best_acc']:.1f}%)", 
                   color=COLORS.get(opt, 'gray'), lw=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Accuracy (%)')
    ax.set_title(f'{model.upper()}')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Section 4.1: Image Classification on ImageNet', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig('fig_classification_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table (like LION paper Table 2)
print("\n" + "="*70)
print("Table 2: ImageNet Classification Results (Top-1 Accuracy %)")
print("="*70)
print(f"{'Model':<12} {'AdamW':<10} {'LION':<10} {'RLO':<10} {'RLO-λA':<10} {'SL-RLO':<10}")
print("-"*70)

for model in models:
    results = classification_results.get(model, {})
    row = [model]
    for opt in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        if opt in results:
            row.append(f"{results[opt]['best_acc']:.2f}")
        else:
            row.append('-')
    print(f"{row[0]:<12} {row[1]:<10} {row[2]:<10} {row[3]:<10} {row[4]:<10} {row[5]:<10}")
print("="*70)

## 2. Section 4.2: Vision-Language (LiT) Results

In [ ]:
# Load LiT results
lit_results = load_results('./results_lit', 'lit_*_results.json')

fig, ax = plt.subplots(figsize=(10, 6))

for opt, data in lit_results.items():
    if 'history' in data and 'loss' in data['history']:
        loss = data['history']['loss']
        ax.plot(loss, label=f"{LABELS.get(opt, opt)} (final={data['final_loss']:.4f})",
               color=COLORS.get(opt, 'gray'), lw=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Contrastive Loss')
ax.set_title('Section 4.2: Vision-Language Contrastive Learning (LiT)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_lit_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Section 4.3: Diffusion Model Results

In [ ]:
# Load diffusion results
diff_64_results = load_results('./results_diffusion', 'diffusion_64_*_results.json')
diff_128_results = load_results('./results_diffusion', 'diffusion_128_*_results.json')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (res, title) in zip(axes, [(diff_64_results, '64×64'), (diff_128_results, '128×128')]):
    for opt, data in res.items():
        if 'history' in data and 'loss' in data['history']:
            loss = data['history']['loss']
            ax.plot(loss, label=f"{LABELS.get(opt, opt)}", color=COLORS.get(opt, 'gray'), lw=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title(f'Diffusion {title}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Section 4.3: Image Generation with Diffusion Models', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('fig_diffusion_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Section 4.4: Language Modeling Results

In [ ]:
# Load LM results
lm_results = load_results('./results_lm', 'lm_*_results.json')

fig, ax = plt.subplots(figsize=(10, 6))

for opt, data in lm_results.items():
    if 'history' in data and 'val_ppl' in data['history']:
        ppl = data['history']['val_ppl']
        ax.plot(ppl, label=f"{LABELS.get(opt, opt)} (best={data['best_ppl']:.1f})",
               color=COLORS.get(opt, 'gray'), lw=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Perplexity')
ax.set_title('Section 4.4: Language Modeling (GPT)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_lm_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Ablation Studies

In [ ]:
# Load ablation results
ABLATION_DIR = './results_ablation'

def load_ablation(name):
    path = Path(ABLATION_DIR) / f"ablation_{name}.json"
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

In [ ]:
# Plot belief coefficient sensitivity
belief_results = load_ablation('belief_coef')

if belief_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    beliefs = []
    accs = []
    for k, v in sorted(belief_results.items()):
        belief = float(k.split('_')[1])
        beliefs.append(belief)
        accs.append(v['best_acc'])
    
    ax.plot(beliefs, accs, 'o-', lw=2, markersize=10, color='#2ca02c')
    ax.axhline(y=max(accs), color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Belief Coefficient (λ_b)')
    ax.set_ylabel('Best Validation Accuracy (%)')
    ax.set_title('Ablation: Effect of Belief Coefficient')
    ax.grid(True, alpha=0.3)
    
    # Annotate best
    best_idx = np.argmax(accs)
    ax.annotate(f'Best: λ_b={beliefs[best_idx]}', 
               xy=(beliefs[best_idx], accs[best_idx]),
               xytext=(beliefs[best_idx]+0.1, accs[best_idx]-0.5),
               fontsize=11)
    
    plt.tight_layout()
    plt.savefig('fig_ablation_belief.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Plot gamma sensitivity
gamma_results = load_ablation('gamma')

if gamma_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    gammas = []
    accs = []
    for k, v in sorted(gamma_results.items(), key=lambda x: float(x[0].split('_')[1])):
        gamma = float(k.split('_')[1])
        gammas.append(gamma)
        accs.append(v['best_acc'])
    
    ax.plot(gammas, accs, 'o-', lw=2, markersize=10, color='#d62728')
    ax.set_xlabel('Gamma (γ)')
    ax.set_ylabel('Best Validation Accuracy (%)')
    ax.set_title('Ablation: Effect of Gamma (Smoothness Parameter)')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('fig_ablation_gamma.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Component analysis
comp_results = load_ablation('components')

if comp_results:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot training curves
    for opt, data in comp_results.items():
        if 'history' in data and 'val_acc' in data['history']:
            val_acc = data['history']['val_acc']
            label = f"{LABELS.get(opt, opt)} ({data['best_acc']:.1f}%)"
            ax.plot(val_acc, label=label, color=COLORS.get(opt, 'gray'), lw=2)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Accuracy (%)')
    ax.set_title('Ablation: Component Analysis - What Makes RLO Better?')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('fig_ablation_components.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print key finding
    if 'rlo' in comp_results and 'rlo_no_belief' in comp_results:
        gain = comp_results['rlo']['best_acc'] - comp_results['rlo_no_belief']['best_acc']
        print(f"\n*** KEY FINDING ***")
        print(f"Belief correction term adds {gain:+.2f}% over LION-equivalent (RLO with belief=0)")
        print(f"This demonstrates the importance of the Lyapunov-derived belief correction.")

## 6. NAIM Mechanism Evidence

In [ ]:
# Plot NAIM statistics from RLO training
def extract_naim_stats(results):
    """Extract NAIM statistics from results"""
    naim_data = {}
    for opt, data in results.items():
        if 'history' in data and 'naim' in data['history']:
            naim = data['history']['naim']
            if naim and len(naim) > 0 and naim[0]:  # Check if naim data exists
                naim_data[opt] = {
                    'belief_norm': [n.get('belief_norm', 0) for n in naim if n],
                    'alignment': [n.get('alignment', 0) for n in naim if n],
                    'grad_norm': [n.get('grad_norm', 0) for n in naim if n],
                }
    return naim_data

# Try to load NAIM data from classification results
naim_data = {}
for model in models:
    if model in classification_results:
        naim_data[model] = extract_naim_stats(classification_results[model])

if any(naim_data.values()):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Find model with NAIM data
    for model, data in naim_data.items():
        if data:
            for opt, naim in data.items():
                if 'rlo' in opt and naim.get('belief_norm'):
                    axes[0].plot(naim['belief_norm'], label=f"{model} - {opt}", lw=2)
                if naim.get('alignment'):
                    axes[1].plot(naim['alignment'], label=f"{model} - {opt}", lw=2)
    
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Belief Norm')
    axes[0].set_title('Belief Correction Magnitude Over Training')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Alignment (cosine)')
    axes[1].set_title('Gradient-Momentum Alignment Over Training')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('NAIM Mechanism Evidence', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig('fig_naim_evidence.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No NAIM data available. Run experiments with track_stats=True.")

## 7. Summary Table (Paper-Ready)

In [ ]:
# Generate comprehensive summary table
print("\n" + "="*90)
print("COMPREHENSIVE RESULTS SUMMARY")
print("="*90)

# Classification
print("\n--- 4.1 Image Classification (Top-1 Accuracy %) ---")
for model in models:
    if model in classification_results:
        results = classification_results[model]
        best_opt = max(results.items(), key=lambda x: x[1]['best_acc'])
        print(f"  {model}: Best = {best_opt[0]} ({best_opt[1]['best_acc']:.2f}%)")

# LiT
print("\n--- 4.2 Vision-Language LiT (Final Loss, lower is better) ---")
if lit_results:
    best_opt = min(lit_results.items(), key=lambda x: x[1]['final_loss'])
    print(f"  Best = {best_opt[0]} ({best_opt[1]['final_loss']:.4f})")

# Diffusion
print("\n--- 4.3 Diffusion (Final Loss, lower is better) ---")
for res, name in [(diff_64_results, '64x64'), (diff_128_results, '128x128')]:
    if res:
        best_opt = min(res.items(), key=lambda x: x[1]['final_loss'])
        print(f"  {name}: Best = {best_opt[0]} ({best_opt[1]['final_loss']:.4f})")

# LM
print("\n--- 4.4 Language Modeling (Perplexity, lower is better) ---")
if lm_results:
    best_opt = min(lm_results.items(), key=lambda x: x[1]['best_ppl'])
    print(f"  Best = {best_opt[0]} ({best_opt[1]['best_ppl']:.2f})")

print("\n" + "="*90)

In [ ]:
# Save all figures list
print("\nGenerated Figures:")
print("  - fig_classification_curves.png")
print("  - fig_lit_curves.png")
print("  - fig_diffusion_curves.png")
print("  - fig_lm_curves.png")
print("  - fig_ablation_belief.png")
print("  - fig_ablation_gamma.png")
print("  - fig_ablation_components.png")
print("  - fig_naim_evidence.png")